# 10 Two Stage Model Optimization

采用 `23` 合并后的 `4` 类风险标签体系，对比单阶段与两阶段风控架构，并输出最终优化报告。


In [ ]:
from __future__ import annotations

import json
from pathlib import Path
import sys

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
try:
    from IPython.display import Markdown, display
except ImportError:
    def Markdown(text):
        return text

    def display(obj):
        print(obj)

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

for candidate in (PROJECT_ROOT, PROJECT_ROOT / "src"):
    candidate_text = str(candidate)
    if candidate_text not in sys.path:
        sys.path.insert(0, candidate_text)

from src.features.feature_selector import CreditFeatureSelector
from src.features.preprocessor import CreditDataPreprocessor
from src.features.sampler import CreditSampler
from src.models.model_evaluator import CreditModelEvaluator
from src.models.risk_classifier import CreditRiskClassifier
from src.models.threshold_adjuster import RiskThresholdAdjuster
from src.models.two_stage_risk_model import TwoStageCreditRiskModel

DATA_DIR = PROJECT_ROOT / "data" / "processed"
MODEL_DIR = PROJECT_ROOT / "src" / "models"
FIGURE_DIR = PROJECT_ROOT / "results" / "figures"
DOCS_DIR = PROJECT_ROOT / "docs"
FIGURE_DIR.mkdir(parents=True, exist_ok=True)
DOCS_DIR.mkdir(parents=True, exist_ok=True)

CLASS_NAMES = ["正常类", "关注类", "次级/可疑类", "损失类"]
MANUAL_CLASS_WEIGHT = {0: 1, 1: 5, 2: 20, 3: 35}

def resolve_split_path(split_name: str) -> Path:
    purified_path = DATA_DIR / f"purified_{split_name}.csv"
    return purified_path if purified_path.exists() else DATA_DIR / f"{split_name}.csv"


In [ ]:
class FixedPredictionModel:
    """Wrap pre-computed predictions and probabilities for evaluation."""

    def __init__(self, y_pred: np.ndarray, y_proba: np.ndarray) -> None:
        self._y_pred = np.asarray(y_pred, dtype=int)
        self._y_proba = np.asarray(y_proba, dtype=float)
        self.classes_ = np.arange(self._y_proba.shape[1])

    def predict(self, X):
        return self._y_pred

    def predict_proba(self, X):
        return self._y_proba


In [ ]:
train_df = pd.read_csv(resolve_split_path("train"), low_memory=False)
val_df = pd.read_csv(resolve_split_path("val"), low_memory=False)
test_df = pd.read_csv(resolve_split_path("test"), low_memory=False)

y_train = train_df["preloan_risk_label"].astype(int)
y_val = val_df["preloan_risk_label"].astype(int)
y_test = test_df["preloan_risk_label"].astype(int)

preprocessor = CreditDataPreprocessor(target_column="preloan_risk_label")
X_train_processed = preprocessor.fit_transform(train_df)
X_val_processed = preprocessor.transform(val_df)
X_test_processed = preprocessor.transform(test_df)

categorical_feature_indices = [
    index
    for index, column in enumerate(X_train_processed.columns)
    if "=" in column and not column.endswith("__target_encoded")
]

display(Markdown("当前脚本已切换到 `4_class_merge_23` 正式方案，并会优先读取 `purified_train/val/test.csv`。"))


In [ ]:
def compute_high_risk_recall(metric_payload: dict) -> float:
    report = metric_payload["classification_report_named"]
    risk_recalls = [
        float(report.get(class_name, {}).get("recall", 0.0))
        for class_name in metric_payload["class_names"]
        if class_name != "正常类"
    ]
    return float(np.mean(risk_recalls)) if risk_recalls else 0.0


def apply_business_rules(frame: pd.DataFrame, predictions: np.ndarray, adjuster: RiskThresholdAdjuster) -> np.ndarray:
    final_predictions = []
    for (_, row), prediction in zip(frame.iterrows(), predictions):
        final_predictions.append(adjuster.business_rule_override(row, int(prediction)))
    return np.asarray(final_predictions, dtype=int)


def evaluate_adjusted_predictions(y_true: pd.Series, y_pred: np.ndarray, y_proba: np.ndarray, X_eval) -> dict:
    wrapped_model = FixedPredictionModel(y_pred, y_proba)
    return CreditModelEvaluator(
        wrapped_model,
        X_eval,
        y_true,
        CLASS_NAMES,
    ).evaluate_imbalanced_multiclass()


def summarize_result(name: str, metrics: dict, extra=None) -> dict:
    report = metrics["classification_report_named"]
    summary = {
        "model_name": name,
        "macro_f1": float(metrics["macro_f1"]),
        "weighted_f1": float(metrics["weighted_f1"]),
        "ks_value": float(metrics["ks_value"]),
        "normal_precision": float(report.get("正常类", {}).get("precision", 0.0)),
        "high_risk_recall": compute_high_risk_recall(metrics),
    }
    if extra:
        summary.update(extra)
    return summary


In [ ]:
selector_baseline = CreditFeatureSelector(top_k_features=80, use_pca=False, n_estimators=100)
X_train_base = selector_baseline.fit_transform(X_train_processed, y_train)
X_val_base = selector_baseline.transform(X_val_processed)
X_test_base = selector_baseline.transform(X_test_processed)

baseline_model = CreditRiskClassifier(model_type="lightgbm", class_weight_mode="none")
baseline_model.fit(X_train_base, y_train)
baseline_metrics = CreditModelEvaluator(
    baseline_model,
    X_test_base,
    y_test,
    CLASS_NAMES,
).evaluate_imbalanced_multiclass()

baseline_summary = summarize_result("原始单阶段4分类", baseline_metrics)


In [ ]:
weighted_model = CreditRiskClassifier(
    model_type="lightgbm",
    class_weight_mode="manual",
    custom_class_weight=MANUAL_CLASS_WEIGHT,
)
weighted_model.fit(X_train_base, y_train)
weighted_val_proba = weighted_model.predict_proba(X_val_base)
weighted_test_proba = weighted_model.predict_proba(X_test_base)

threshold_adjuster = RiskThresholdAdjuster(normal_class_threshold=0.9)
best_threshold, _ = threshold_adjuster.find_optimal_threshold(
    weighted_val_proba,
    y_val,
    min_normal_precision=0.8,
)
weighted_threshold_pred = threshold_adjuster.adjust_prediction(weighted_test_proba)
weighted_rule_pred = apply_business_rules(test_df, weighted_threshold_pred, threshold_adjuster)
weighted_rule_metrics = evaluate_adjusted_predictions(
    y_test,
    weighted_rule_pred,
    weighted_test_proba,
    X_test_base,
)
weighted_rule_summary = summarize_result(
    "单阶段+权重+阈值规则",
    weighted_rule_metrics,
    extra={"normal_threshold": float(best_threshold)},
)


In [ ]:
optimized_sampler = CreditSampler(categorical_features=categorical_feature_indices)
X_train_sampled, y_train_sampled = optimized_sampler.fit_resample(
    X_train_processed,
    y_train,
    sampling_strategy="auto",
    max_oversample_ratio=5,
)
optimized_sampler.plot_sample_distribution(y_train, y_train_sampled)

selector_sampled = CreditFeatureSelector(top_k_features=80, use_pca=False, n_estimators=100)
X_train_sampled_selected = selector_sampled.fit_transform(X_train_sampled, y_train_sampled)
X_val_sampled_selected = selector_sampled.transform(X_val_processed)
X_test_sampled_selected = selector_sampled.transform(X_test_processed)

sampled_model = CreditRiskClassifier(
    model_type="lightgbm",
    class_weight_mode="manual",
    custom_class_weight=MANUAL_CLASS_WEIGHT,
)
sampled_model.fit(X_train_sampled_selected, y_train_sampled)
sampled_val_proba = sampled_model.predict_proba(X_val_sampled_selected)
sampled_test_proba = sampled_model.predict_proba(X_test_sampled_selected)

sampled_adjuster = RiskThresholdAdjuster(normal_class_threshold=0.9)
sampled_best_threshold, _ = sampled_adjuster.find_optimal_threshold(
    sampled_val_proba,
    y_val,
    min_normal_precision=0.8,
)
sampled_threshold_pred = sampled_adjuster.adjust_prediction(sampled_test_proba)
sampled_rule_pred = apply_business_rules(test_df, sampled_threshold_pred, sampled_adjuster)
sampled_rule_metrics = evaluate_adjusted_predictions(
    y_test,
    sampled_rule_pred,
    sampled_test_proba,
    X_test_sampled_selected,
)
sampled_rule_summary = summarize_result(
    "单阶段+权重+采样+阈值规则",
    sampled_rule_metrics,
    extra={"normal_threshold": float(sampled_best_threshold)},
)


In [ ]:
two_stage_selector = CreditFeatureSelector(top_k_features=80, use_pca=False, n_estimators=100)
X_train_two_stage = two_stage_selector.fit_transform(X_train_processed, y_train)
X_test_two_stage = two_stage_selector.transform(X_test_processed)

two_stage_model = TwoStageCreditRiskModel()
two_stage_model.fit(X_train_two_stage, y_train)

two_stage_test_pred = two_stage_model.predict_with_threshold_and_rule(
    X_test_two_stage,
    user_data=test_df,
    normal_threshold=two_stage_model.get_threshold(),
)
two_stage_test_proba = two_stage_model.predict_proba(X_test_two_stage)
two_stage_metrics = evaluate_adjusted_predictions(
    y_test,
    two_stage_test_pred,
    two_stage_test_proba,
    X_test_two_stage,
)
two_stage_summary = summarize_result(
    "两阶段分级+权重+阈值规则",
    two_stage_metrics,
    extra={"normal_threshold": float(two_stage_model.get_threshold())},
)

joblib.dump(two_stage_model, MODEL_DIR / "two_stage_risk_model.joblib")


In [ ]:
comparison_df = pd.DataFrame([
    baseline_summary,
    weighted_rule_summary,
    sampled_rule_summary,
    two_stage_summary,
])
display(comparison_df)


In [ ]:
radar_metrics = ["macro_f1", "high_risk_recall", "normal_precision", "ks_value"]
radar_labels = ["Macro-F1", "高风险平均召回率", "正常类精准率", "KS值"]
angles = np.linspace(0, 2 * np.pi, len(radar_metrics), endpoint=False).tolist()
angles += angles[:1]

fig, ax = plt.subplots(figsize=(8, 8), subplot_kw={"polar": True})
for _, row in comparison_df.iterrows():
    values = [float(row[metric]) for metric in radar_metrics]
    values += values[:1]
    ax.plot(angles, values, linewidth=2, label=row["model_name"])
    ax.fill(angles, values, alpha=0.08)

ax.set_xticks(angles[:-1])
ax.set_xticklabels(radar_labels)
ax.set_ylim(0.0, 1.0)
ax.set_title("最终模型核心指标对比雷达图")
ax.legend(loc="upper right", bbox_to_anchor=(1.35, 1.1))

radar_path = FIGURE_DIR / "final_model_comparison_radar.png"
fig.tight_layout()
fig.savefig(radar_path, dpi=200, bbox_inches="tight")
plt.close(fig)

display(Markdown(f"雷达图已保存到 `results/figures/{radar_path.name}`"))


In [ ]:
best_model_row = comparison_df.sort_values(
    by=["macro_f1", "high_risk_recall", "normal_precision"],
    ascending=False,
).iloc[0]

report_lines = [
    "# 最终模型优化报告",
    "",
    "## 架构结论",
    f"- 推荐模型：`{best_model_row['model_name']}`",
    f"- 最优 Macro-F1：`{best_model_row['macro_f1']:.6f}`",
    f"- 高风险类平均召回率：`{best_model_row['high_risk_recall']:.6f}`",
    f"- 正常类精准率：`{best_model_row['normal_precision']:.6f}`",
    "",
    "## 关键配置",
    f"- 单阶段手动权重：`{MANUAL_CLASS_WEIGHT}`",
    f"- 单阶段最优阈值：`{best_threshold:.2f}`",
    f"- 采样方案：`SMOTENC + ENN`，`max_oversample_ratio=5`",
    f"- 两阶段模型阈值：`{two_stage_model.get_threshold():.2f}`",
    "",
    "## 方案说明",
    "- 当前正式标签方案为 `4_class_merge_23`。",
    "- 其中原 `类2` 与 `类3` 已合并为 `次级/可疑类`。",
    "",
    "## 指标对比",
    comparison_df.to_markdown(index=False),
    "",
    "## 图表输出",
    "- `results/figures/sample_distribution_comparison.png`",
    "- `results/figures/final_model_comparison_radar.png`",
    "",
    "## 额外说明",
    "- 原始单阶段模型用于评估未做任何不平衡优化时的基线。",
    "- 两阶段模型将正常类识别与风险细分解耦，更适合当前 `4` 类不平衡任务。",
]

report_path = DOCS_DIR / "final_model_optimization_report.md"
report_path.write_text("\n".join(report_lines), encoding="utf-8")
display(Markdown(f"最终报告已保存到 `docs/{report_path.name}`"))
